# 모델 성능 향상과 앙상블 기법

이번 강의에서는 다음 세 가지 주제를 다룹니다.

1. **과대적합(Overfitting)과 과소적합(Underfitting)**
2. **교차검증(Cross-Validation)의 이해**
3. **앙상블 학습: 랜덤 포레스트(Random Forest)**

In [ ]:
# 공통 라이브러리 임포트
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 재현성을 위한 시드 고정
SEED = 42
np.random.seed(SEED)

---
## 1. 과대적합(Overfitting)과 과소적합(Underfitting)

| 현상 | 설명 | 증상 |
|------|------|------|
| 과소적합 | 모델이 너무 단순해 학습 데이터도 제대로 학습하지 못함 | Train 정확도 ↓, Test 정확도 ↓ |
| 적정 모델 | 학습 데이터와 새로운 데이터 모두 잘 일반화 | Train 정확도 ↑, Test 정확도 ↑ |
| 과대적합 | 모델이 학습 데이터를 너무 외워 새로운 데이터에 일반화 실패 | Train 정확도 ↑↑, Test 정확도 ↓ |

결정 트리의 **max_depth** 파라미터로 과대/과소적합을 직접 확인해봅니다.

In [ ]:
# 데이터 준비
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# max_depth 변화에 따른 Train/Test 정확도 측정
depths = range(1, 20)
train_scores, test_scores = [], []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    model.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test)))

# 시각화
plt.figure(figsize=(9, 5))
plt.plot(depths, train_scores, 'o-', label='Train Accuracy', color='steelblue')
plt.plot(depths, test_scores, 's--', label='Test Accuracy', color='tomato')
plt.axvline(x=depths[np.argmax(test_scores)], color='gray', linestyle=':', alpha=0.7,
            label=f'Best depth = {depths[np.argmax(test_scores)]}')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('과대적합 vs 과소적합: Decision Tree max_depth 변화')
plt.legend()
plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(test_scores)]
print(f"최적 depth: {best_depth}")
print(f"  Train Accuracy: {train_scores[best_depth - 1]:.4f}")
print(f"  Test  Accuracy: {test_scores[best_depth - 1]:.4f}")

### 관찰 포인트
- depth가 낮을 때(1~2): 두 곡선이 모두 낮음 → **과소적합**
- depth가 중간일 때: Test 정확도가 최고 → **적정 복잡도**
- depth가 깊을 때: Train은 계속 오르지만 Test는 하락 → **과대적합**

---
## 2. 교차검증(Cross-Validation)의 이해

단순 홀드아웃(Hold-out) 분리는 **운에 따라 결과가 달라질 수 있습니다.**  
교차검증은 데이터를 여러 폴드(fold)로 나눠 **모든 샘플이 한 번씩 검증 데이터**가 되도록 합니다.

```
K-Fold (K=5) 예시
┌─────┬─────┬─────┬─────┬─────┐
│ Val │Train│Train│Train│Train│  Fold 1
│Train│ Val │Train│Train│Train│  Fold 2
│Train│Train│ Val │Train│Train│  Fold 3
│Train│Train│Train│ Val │Train│  Fold 4
│Train│Train│Train│Train│ Val │  Fold 5
└─────┴─────┴─────┴─────┴─────┘
```

In [ ]:
# ── 2-1. KFold vs StratifiedKFold 비교 ──
model = DecisionTreeClassifier(max_depth=5, random_state=SEED)

kf  = KFold(n_splits=5, shuffle=True, random_state=SEED)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

kf_scores  = cross_val_score(model, X, y, cv=kf,  scoring='accuracy')
skf_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print("[KFold 교차검증]")
print(f"  각 폴드 정확도: {kf_scores.round(4)}")
print(f"  평균: {kf_scores.mean():.4f}  |  표준편차: {kf_scores.std():.4f}")

print("\n[StratifiedKFold 교차검증]")
print(f"  각 폴드 정확도: {skf_scores.round(4)}")
print(f"  평균: {skf_scores.mean():.4f}  |  표준편차: {skf_scores.std():.4f}")

In [ ]:
# ── 2-2. 폴드 수(K) 변화에 따른 안정성 시각화 ──
k_values = [3, 5, 7, 10]
means, stds = [], []

for k in k_values:
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    means.append(scores.mean())
    stds.append(scores.std())

plt.figure(figsize=(8, 5))
plt.errorbar(k_values, means, yerr=stds, fmt='o-', capsize=6,
             color='mediumseagreen', ecolor='salmon', linewidth=2)
plt.xlabel('K (폴드 수)')
plt.ylabel('Accuracy')
plt.title('K 값에 따른 교차검증 정확도 (평균 ± 표준편차)')
plt.xticks(k_values)
plt.tight_layout()
plt.show()

### 핵심 정리
- **KFold**: 클래스 비율을 고려하지 않음 → 불균형 데이터에서 불안정
- **StratifiedKFold**: 각 폴드의 클래스 비율 유지 → **분류 문제에 권장**
- K가 클수록 분산은 줄지만 계산 비용 증가 → 보통 **K=5 또는 K=10** 사용

---
## 3. 앙상블 학습: 랜덤 포레스트(Random Forest)

앙상블(Ensemble)은 **여러 모델의 예측을 결합**해 단일 모델보다 높은 성능을 얻는 기법입니다.

### 랜덤 포레스트 동작 원리

```
원본 데이터
    │
    ├──[부트스트랩 샘플 1] → 결정 트리 1 → 예측 1 ┐
    ├──[부트스트랩 샘플 2] → 결정 트리 2 → 예측 2 ├─ 다수결 → 최종 예측
    ⋮                                             ⋮
    └──[부트스트랩 샘플 N] → 결정 트리 N → 예측 N ┘
```

두 가지 **무작위성** 주입으로 다양성 확보:
1. **배깅(Bagging)**: 복원 추출로 부트스트랩 샘플 생성
2. **특성 무작위 선택**: 각 분기점에서 일부 특성만 고려

In [ ]:
# ── 3-1. 단일 결정 트리 vs 랜덤 포레스트 성능 비교 ──
dt  = DecisionTreeClassifier(random_state=SEED)
rf  = RandomForestClassifier(n_estimators=100, random_state=SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

dt_scores = cross_val_score(dt, X, y, cv=cv, scoring='accuracy')
rf_scores = cross_val_score(rf, X, y, cv=cv, scoring='accuracy')

print("[단일 결정 트리]")
print(f"  각 폴드: {dt_scores.round(4)}")
print(f"  평균: {dt_scores.mean():.4f}  |  표준편차: {dt_scores.std():.4f}")

print("\n[랜덤 포레스트 (n=100)]")
print(f"  각 폴드: {rf_scores.round(4)}")
print(f"  평균: {rf_scores.mean():.4f}  |  표준편차: {rf_scores.std():.4f}")

In [ ]:
# ── 3-2. 트리 수(n_estimators)에 따른 성능 변화 ──
n_trees = [1, 5, 10, 20, 50, 100, 200]
rf_means, rf_stds = [], []

for n in n_trees:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=SEED)
    scores = cross_val_score(rf_n, X, y, cv=cv, scoring='accuracy')
    rf_means.append(scores.mean())
    rf_stds.append(scores.std())

plt.figure(figsize=(9, 5))
plt.errorbar(n_trees, rf_means, yerr=rf_stds, fmt='o-', capsize=5,
             color='darkorange', ecolor='gray', linewidth=2)
plt.axhline(y=dt_scores.mean(), color='steelblue', linestyle='--',
            label=f'단일 결정 트리 평균 ({dt_scores.mean():.4f})')
plt.xlabel('트리 수 (n_estimators)')
plt.ylabel('CV Accuracy')
plt.title('랜덤 포레스트: 트리 수에 따른 교차검증 정확도')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 3-3. 특성 중요도(Feature Importance) 시각화 ──
rf_final = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_final.fit(X_train, y_train)

importances = rf_final.feature_importances_
indices = np.argsort(importances)[::-1][:15]  # 상위 15개
feature_names = cancer.feature_names

plt.figure(figsize=(10, 6))
plt.bar(range(len(indices)), importances[indices], color='mediumpurple', edgecolor='black')
plt.xticks(range(len(indices)), feature_names[indices], rotation=45, ha='right')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('랜덤 포레스트 특성 중요도 (상위 15개)')
plt.tight_layout()
plt.show()

test_acc = accuracy_score(y_test, rf_final.predict(X_test))
print(f"최종 Test Accuracy: {test_acc:.4f}")

### 핵심 정리

| 항목 | 내용 |
|------|------|
| 과대적합 방지 | 여러 트리가 서로 다른 패턴을 학습 → 평균 내면 분산 감소 |
| 성능 수렴 | 트리가 충분히 많아지면(~100) 이후로는 큰 개선 없음 |
| 특성 중요도 | 불순도 감소 기여도로 각 특성의 중요도 자동 계산 |
| 주요 하이퍼파라미터 | `n_estimators`, `max_depth`, `max_features`, `min_samples_split` |

---

## 전체 요약

| 개념 | 핵심 메시지 |
|------|-------------|
| 과대/과소적합 | 모델 복잡도를 조정해 Train-Test 정확도 간격 최소화 |
| 교차검증 | 단일 분리 대신 K번 반복 평가로 신뢰도 높은 성능 추정 |
| 랜덤 포레스트 | 배깅 + 특성 무작위성으로 다양한 트리 앙상블 → 일반화 성능 향상 |